In [1]:
import pandas as pd

In [2]:



#Taking a sample because dealing with 99 million rows of data at once is computationally heavy

import pyarrow as pa
import pyarrow.parquet as pq

source = pq.ParquetFile("Data/2026-09.parquet")
output = "Data/sample_100k.parquet"

remaining = 100_000

with pq.ParquetWriter(output, source.schema_arrow) as writer:
    for batch in source.iter_batches(batch_size=10_000):
        batch = batch.slice(0, min(batch.num_rows, remaining))

        writer.write_table(pa.Table.from_batches([batch]))
        remaining -= batch.num_rows

        print(f"Remaining: {remaining:,}")

        if remaining == 0:
            break

print("Sample saved!")

Remaining: 90,000
Remaining: 80,000
Remaining: 70,000
Remaining: 60,000
Remaining: 50,000
Remaining: 40,000
Remaining: 30,000
Remaining: 20,000
Remaining: 10,000
Remaining: 0
Sample saved!


In [3]:
df = pd.read_parquet('Data/sample_100k.parquet')

In [4]:
df['is_ai_generated'].sum() #5000 out of 120k are ai 

np.float64(5091.0)

In [5]:
df.dtypes

video_id                                uint64
posted_at                  datetime64[ms, UTC]
country                                    str
language                                   str
type                                       str
duration_ms                             uint32
image_count                              uint8
width                                   uint16
height                                  uint16
caption                                    str
hashtags                                object
hashtag_ids                             object
mentions                                object
on_screen_text                          object
sound_id                                uint64
is_ad                                    uint8
branded_content                            str
product_id                              uint64
is_shop_video                            uint8
seller_id                               uint64
is_ai_generated                        float64
not_recommend

In [6]:
print(df['language'].unique())
print(len(df[df['language'] == 'un']))

<ArrowStringArray>
[ 'un',  'en',  'ar',  'km',  'nl',  'ru',  'id',  'es',  'sr',  'de',
 ...
  'tw',  'kn', 'kmr',  'ga', 'nso',  'ti',  'be',  'to',  'hy',  'gd']
Length: 114, dtype: str
58662


In [7]:
dl = df.groupby('language').mean(numeric_only=True)

In [8]:
stats = (df.groupby('language')['is_ai_generated']
           .agg(rate='mean', n='size', labeled='count')
           .sort_values('n', ascending=False))

stats[stats['labeled'] >= 30].sort_values('rate', ascending=False)



,rate,n,labeled
language,,,
zh-Hans,0.223684,93,76
pl,0.179104,81,67
ta,0.169811,59,53
om,0.121212,80,66
ro,0.102564,48,39
nl,0.102564,46,39
it,0.100917,124,109
ne,0.096059,508,406
si,0.093333,174,150


In [9]:
import numpy as np
df['caption'] = df['caption'].str.strip().replace('', np.nan)

In [10]:
#we would like to take the un and fill it using the caption since it has much data about the language of the video 
df['caption'].isna().sum()
df[(df['language'] == 'un') & (df['caption'].isna())]

,video_id,posted_at,country,language,type,duration_ms,image_count,width,height,caption,...,not_recommended_for_fyp,sound_muted,views,likes,comments,shares,saves,downloads,engagement_rate,stats_updated_at
5,7680354140752497934,2026-09-01 00:17:01+00:00,--,un,video,10000,0,0,0,NaN,...,0,0,283,38,1,0,0,0,0.137809,2026-09-17 23:07:58+00:00
6,7680354325939473677,2026-09-01 00:17:42+00:00,--,un,video,15000,0,0,0,NaN,...,0,0,346,43,0,0,0,0,0.124277,2026-09-23 23:59:59+00:00
12,7680363720639253790,2026-09-01 00:54:00+00:00,--,un,video,15046,0,1080,1920,NaN,...,0,0,347,35,1,0,4,0,0.115274,2026-09-30 00:13:02+00:00
14,7680364013917621535,2026-09-01 00:55:10+00:00,--,un,video,13000,0,0,0,NaN,...,0,0,269,3,0,0,0,0,0.011152,2026-09-18 06:46:37+00:00
15,7680366081042648340,2026-09-01 01:03:07+00:00,--,un,video,57234,0,720,1280,NaN,...,0,0,168,21,4,1,2,1,0.166667,2026-09-30 01:12:41+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99984,7682349797663149320,2026-09-06 09:20:54+00:00,AE,un,photo,0,1,0,0,NaN,...,0,0,820,33,0,3,1,0,0.045122,2026-10-03 09:07:41+00:00
99986,7682349811001085191,2026-09-06 09:21:03+00:00,AE,un,video,13000,0,0,0,NaN,...,0,0,75,4,0,0,0,0,0.053333,2026-09-10 03:09:36+00:00
99992,7682349895788793106,2026-09-06 09:21:22+00:00,AE,un,video,16000,0,0,0,NaN,...,0,0,84,6,0,0,0,0,0.071429,2026-09-10 03:09:36+00:00
99995,7682349933487213842,2026-09-06 09:21:28+00:00,AE,un,video,15091,0,1080,1920,NaN,...,0,0,450,129,13,0,5,0,0.326667,2026-10-03 09:07:41+00:00


In [11]:

from lingua import LanguageDetectorBuilder

detector = LanguageDetectorBuilder.from_all_languages().build()

mask = (df['language'] == 'un') & df['caption'].notna()

# Get all captions that need detection
texts = df.loc[mask, 'caption'].tolist()

# Detect languages in parallel
results = detector.compute_language_confidence_values_in_parallel(texts)

# Convert results to language codes
def get_code(result):
    if not result or result[0].value < 0.5:
        return 'un'

    code = result[0].language.iso_code_639_1
    return code.name.lower() if code else 'un'

df.loc[mask, 'language'] = [get_code(r) for r in results]

print("Remaining unknown:", (df['language'] == 'un').sum())

Remaining unknown: 50773


In [12]:
#NOTE ONLY AROUND 7K ENTRIES GOT FILLED


In [13]:
#frequency encoding for the data 
freq = df['language'].value_counts(normalize=True)   # each language -> its share of rows
df['lang_freq'] = df['language'].map(freq)           # replace each row's language with that share
df = df.drop(columns=['language'])

In [14]:
df.groupby('not_recommended_for_fyp')['is_ai_generated'].mean()

not_recommended_for_fyp
0    0.065107
1    0.098485
Name: is_ai_generated, dtype: float64

In [15]:
df['branded_content'].unique()

df['branded_content'].value_counts()

branded_content
                    99305
paid_partnership      686
shop_affiliate          9
Name: count, dtype: int64

In [16]:
df['is_branded'] = ((df['branded_content'] == 'paid_partnership') |
                    (df['branded_content'] == 'shop_affiliate')).astype(int)

df = df.drop(columns=['branded_content'])

In [17]:
df['sound_id'].unique() #analysis done on the whole dataset its better to keep this if we were training on the whole dataset but since we took only a part of it for this month 
#and its mostly 1.5 IDs for each row which gives us very little signal we drop for the full dataset we can since the ids repeate more 

df.drop(columns = 'sound_id', inplace = True)

In [18]:
df['n_hashtags'] = df['hashtags'].apply(lambda x: len(x) if x is not None else 0)
df = df.drop(columns=['hashtag_ids'])

In [19]:
freq = df['country'].value_counts(normalize=True)
freq
df['country_freq'] = df['country'].map(freq)
df.drop(columns='country', inplace = True, errors= 'ignore')

In [20]:
df.iloc[20:30]
#hash tags and caption gives similar data we drop hashtags
df.drop(columns='hashtags',inplace=True)

In [21]:
df.isna().sum()

video_id                       0
posted_at                      0
type                           0
duration_ms                    0
image_count                    0
width                          0
height                         0
caption                    24015
mentions                       0
on_screen_text                 0
is_ad                          0
product_id                     0
is_shop_video                  0
seller_id                      0
is_ai_generated            22076
not_recommended_for_fyp        0
sound_muted                    0
views                          0
likes                          0
comments                       0
shares                         0
saves                          0
downloads                      0
engagement_rate              518
stats_updated_at               0
lang_freq                      0
is_branded                     0
n_hashtags                     0
country_freq                   0
dtype: int64

In [22]:
df['product_id'].unique()

array([                  0, 1729436595308040448, 1736256815044265160,
       1729388433496379544, 1730921538978351393, 1731778648852957013,
       1730517716918635382, 1732211811415856062, 1731255609618961080,
       1732349606004166816, 1732501527579363069, 1732300587529962730,
       1732602266995626602, 1736171586173764922], dtype=uint64)

In [23]:
df = df.drop(columns=['product_id', 'seller_id'])

In [24]:
import ast
import numpy as np

def parse_mentions(x):
    if isinstance(x, str):
        return ast.literal_eval(x)
    return x

df['mentions'] = df['mentions'].apply(parse_mentions)

In [25]:
df['mentions'].apply(lambda x: len(x) == 0).sum()

np.int64(90248)

In [26]:
df['has_mentions'] = df['mentions'].apply(lambda x: int(len(x) > 0))

In [27]:
df.drop(columns=['mentions','video_id'], inplace= True, errors= 'ignore')

In [28]:
x = pd.get_dummies(df['type'],
               prefix='type_')
x

,type__photo,type__video
0,False,True
1,False,True
2,False,True
3,False,True
4,False,True
...,...,...
99995,False,True
99996,False,True
99997,False,True
99998,False,True


In [29]:
df = df.drop(columns=['type'])
df['type_video'] = x['type__video'].astype('int8')

In [30]:
df['update_delay_min'] = (
    df['stats_updated_at'] - df['posted_at']
).dt.total_seconds() / 60
df.drop(columns=['stats_updated_at','posted_at'],inplace= True, errors='ignore')

In [31]:
df['on_screen_text_length'] = (
    df['on_screen_text'].fillna('').str.strip().str.len()
)

df['has_on_screen_text'] = (
    df['on_screen_text_length'] > 0
).astype('uint8')

AttributeError: Can only use .str accessor with string values, not floating

In [33]:
df['on_screen_text']

0        [Let's Take a Trip Back To 1989... \nThe Last ...
1        [POP 2K:\nIt's Not You,\nIt's Me Edition 🤔, I ...
2                                                       []
3                                                       []
4                                                       []
                               ...                        
99995                                                   []
99996                                                   []
99997                           [On this day, 6 Sept 2024]
99998                                                   []
99999                                                   []
Name: on_screen_text, Length: 100000, dtype: object

In [34]:
df['on_screen_text_length'] = df['on_screen_text'].apply(
    lambda texts: sum(len(t.strip()) for t in texts if isinstance(t, str))
)

In [35]:
df.drop(columns='on_screen_text', inplace=True)

In [37]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cuda"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Moha\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Moha\.cache\huggingface\hub\models--Qwen--Qwen3-Embedding-0.6B. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

In [ ]:
#test
captions = df['caption'].fillna('').astype(str).tolist()[:1000]
embeddings = model.encode(
    captions,
    batch_size=32,
    prompt_name="document",
    normalize_embeddings=True,
    truncate_dim=512,
    convert_to_numpy=True,
    show_progress_bar=True
)
print(embeddings.shape)
##taking so long need better opitimization 

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

KeyboardInterrupt: 